# TaaS Model & Agentic Studio — Colab T4 Quick Launch

**Operator guide:** [Integrated Colab quickstart](../../docs/deployment/COLAB_T4_QUICKSTART.md)

This notebook launches the integrated Model/Agent/TaaS Studio from an immutable repository revision inside the current Colab runtime.

**Available in this pinned revision:** Hugging Face model search and metadata inspection, explicit model loading, real model-backed chat inference, model-assisted GoalSpec proposals, TaaS tool discovery and pre-execution assurance, and configured MCP stdio server/tool inspection.

**Important boundaries:** MCP inspection does not invoke tools; TaaS certification is pre-execution evidence; the Agent workspace does not yet run an autonomous tool-calling loop. Persistent run history, streaming traces, multi-user isolation and production deployment are not provided.

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, secrets
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "4a3e4e460f2556f6556b1ef4736f991337cf32c2"
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
assert (WORKDIR / "pyproject.toml").is_file()
assert (WORKDIR / "src" / "taas").is_dir()
print("TaaS revision:", actual)

## Hardware detection

A T4 is useful for local model inference when Colab allocates one. The TaaS discovery engine itself is CPU-oriented and does not require a GPU.

The model runtime loads weights in the same Python process that hosts this Studio API. A successful GPU probe is not a model-load test; verify actual load and inference from Model Studio. If no GPU is available, CPU mode may still work for sufficiently small models.

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; CPU discovery mode.")
print("Python:", sys.version.split()[0])

## Install and verify TaaS Studio

Install the Studio API and optional Hugging Face inference dependencies into the active Colab interpreter. The model runtime requires the `hf` extra (Transformers, PyTorch, Accelerate and Safetensors). If installation or import verification fails, the notebook stops with the diagnostic output.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[studio,hf]"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(install.stdout[-4000:])
if install.returncode:
    print(install.stderr[-4000:])
    raise RuntimeError("TaaS Studio installation failed")
import_check = subprocess.run(
    [
        sys.executable, "-c",
        "import traceback, importlib; "
        "importlib.import_module('taas'); "
        "importlib.import_module('fastapi'); "
        "importlib.import_module('uvicorn'); "
        "importlib.import_module('torch'); "
        "importlib.import_module('transformers'); "
        "importlib.import_module('safetensors'); "
        "importlib.import_module('taas.studio.api'); "
        "print('TaaS Model/Agent Studio imports OK')"
    ],
    cwd=WORKDIR, text=True, capture_output=True
)
print(import_check.stdout)
if import_check.returncode:
    print("=== IMPORT CHECK STDERR / TRACEBACK ===")
    print(import_check.stderr[-12000:])
    print("Python executable:", sys.executable)
    print("Working directory:", WORKDIR)
    print("Package source exists:", (WORKDIR / "src" / "taas" / "__init__.py").is_file())
    raise RuntimeError("TaaS Model/Agent Studio import check failed; see traceback above")

## Launch Studio

An ephemeral API token is generated automatically for this Colab runtime. Copy it into the Studio token field; it is not committed or persisted by this notebook.

The default TaaS model profile remains **strict and uncalibrated** unless an operator explicitly supplies a measured `TAAS_MODEL_PROFILE`. Installing or loading a Hugging Face model does not manufacture capability calibration. Unknown native capabilities therefore remain unassumed.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
print("Temporary Studio API token (copy into Studio; do not share):")
print(api_token)

# Make rerunning this cell safe, including after the Stop cell was used.
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.close()

studio_log = open("/content/taas-studio.log", "a", buffering=1)
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_BIND_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=40):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health"))
print("Readiness:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/ready"))
print("Studio process PID:", studio_process.pid)


## Open TaaS Studio

Run this cell and open the link. Keep the Colab runtime connected while using Studio.

In [ ]:
from google.colab import output
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))

## Stop Studio

Run this cell when finished. Colab storage and runtime memory are temporary.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.flush()
    studio_log.close()
print("TaaS Studio stopped.")
